In [312]:
import pandas as pd
import numpy as np
import re
import os
import json
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [313]:
DATA_PATH = "../data/processed/"
OUTPUT_PATH = "../outputs/"

In [314]:
with open(
    DATA_PATH + "nvidia_research_corpus.txt",
    "r",
    encoding="utf-8"
) as f:
    research_corpus = f.read()

In [315]:
print("Corpus characters:", len(research_corpus))

Corpus characters: 534947


In [316]:
research_corpus = re.sub(
    r"\s+",
    " ",
    research_corpus
).strip()

In [317]:
words = research_corpus.split()

chunk_size = 150
overlap = 30

chunks = []

start = 0

while start < len(words):
    end = start + chunk_size
    
    chunk = " ".join(
        words[start:end]
    )
    
    chunks.append(chunk)
    
    start += chunk_size - overlap

In [318]:
rag_chunks = pd.DataFrame({
    "chunk_id": range(len(chunks)),
    "text": chunks
})

In [319]:
rag_chunks.head()

,chunk_id,text
0,0,NVIDIA 10-K nvda-20260125 0001045810 2026 FY f...
1,1,2025-01-26 0001045810 us-gaap:AccumulatedOther...
2,2,0001045810 us-gaap:EmployeeStockMember 2024-01...
3,3,us-gaap:USGovernmentAgenciesDebtSecuritiesMemb...
4,4,us-gaap:USGovernmentAgenciesDebtSecuritiesMemb...


In [320]:
print("Total chunks:", len(rag_chunks))

Total chunks: 649


In [321]:
rag_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

rag_matrix = rag_vectorizer.fit_transform(
    rag_chunks["text"]
)

In [322]:
print("RAG matrix shape:", rag_matrix.shape)

RAG matrix shape: (649, 15359)


In [323]:
rag_chunks.to_csv(
    DATA_PATH + "nvidia_rag_chunks.csv",
    index=False
)

In [324]:
joblib.dump(
    rag_vectorizer,
    OUTPUT_PATH + "nvidia_rag_tfidf_vectorizer.pkl"
)

['../outputs/nvidia_rag_tfidf_vectorizer.pkl']

In [325]:
from scipy.sparse import save_npz

save_npz(
    OUTPUT_PATH + "nvidia_rag_matrix.npz",
    rag_matrix
)

In [326]:
def retrieve_documents(
    query,
    vectorizer,
    matrix,
    chunks,
    top_k=5
):
    query_vector = vectorizer.transform(
        [query]
    )
    
    similarities = cosine_similarity(
        query_vector,
        matrix
    )[0]
    
    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]
    
    results = chunks.iloc[
        top_indices
    ].copy()
    
    results["Similarity"] = similarities[
        top_indices
    ]
    
    return results

In [327]:
query = "What are NVIDIA's major risks?"

retrieved_results = retrieve_documents(
    query,
    rag_vectorizer,
    rag_matrix,
    rag_chunks,
    top_k=5
)

retrieved_results

,chunk_id,text,Similarity
44,44,center-scale compute and networking solutions ...,0.088827
594,594,"our legal proceedings since January 25, 2026. ...",0.088241
28,28,design � for real-time trillion-parameter infe...,0.080852
45,45,AI agents while preserving privacy. These tool...,0.074330
35,35,enable hyperscalers and custom ASIC designers ...,0.069917


In [328]:
for _, row in retrieved_results.iterrows():
    print(
        f"\nSimilarity: {row['Similarity']:.4f}"
    )
    print(row["text"])


Similarity: 0.0888
center-scale compute and networking solutions across processing units, interconnects, systems, and software. Our compute solutions include all three major processing units in AI servers � GPUs, CPUs, and DPUs. GPUs are uniquely suited to AI, and we will continue to add AI-specific features to our GPU architecture to further extend our leadership position. In addition, we offer NVIDIA AI Enterprise�a comprehensive software suite designed to simplify the development and deployment of production-grade, end-to-end generative AI applications. NVIDIA AI Enterprise includes: NVIDIA NIM, which increases token throughput using industry-leading open and proprietary models; NVIDIA NeMo, a complete solution for curating, fine-tuning, reinforcement learning, evaluating, and safeguarding domain-adapted models; and AI Blueprints, pre-built, runnable templates that help enterprises build, optimize, and deploy AI agents while preserving privacy. These tools enable organizations to s

In [329]:
def rag_search(query, top_k=5):
    results = retrieve_documents(
        query,
        rag_vectorizer,
        rag_matrix,
        rag_chunks,
        top_k=top_k
    )

    return results

In [330]:
queries = [
    "What does NVIDIA do?",
    "What are NVIDIA's major products?",
    "What are NVIDIA's main business risks?",
    "How important is artificial intelligence to NVIDIA?"
]

for query in queries:
    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = rag_search(
        query,
        top_k=3
    )

    for _, row in results.iterrows():
        print(
            f"\nSimilarity: {row['Similarity']:.4f}"
        )
        print(row["text"][:1000])


QUERY: What does NVIDIA do?

Similarity: 0.0809
( 27,386 ) Change in cash and cash equivalents 11,838 3,050 Cash and cash equivalents at beginning of period 10,605 8,589 Cash and cash equivalents at end of period $ 22,443 $ 11,639 See accompanying Notes to Condensed Consolidated Financial Statements. 8 NVIDIA Corporation and Subsidiaries Notes to Condensed Consolidated Financial Statements (Unaudited) Note 1 - Summary of Significant Accounting Policies Basis of Presentation The accompanying unaudited condensed consolidated financial statements were prepared in accordance with accounting principles generally accepted in the United States of America, or U.S. GAAP, for interim financial information and with the instructions to Form 10-Q and Article 10 of Securities and Exchange Commission, or SEC, Regulation S-X. The January 25, 2026, consolidated balance sheet was derived from our audited consolidated financial statements included in our Annual Report on Form 10-K for the fiscal year en

In [331]:
joblib.dump(
    rag_vectorizer,
    OUTPUT_PATH + "nvidia_rag_vectorizer.pkl"
)

['../outputs/nvidia_rag_vectorizer.pkl']

In [332]:
loaded_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_rag_vectorizer.pkl"
)

loaded_matrix = rag_matrix
loaded_chunks = rag_chunks

In [333]:
test_results = retrieve_documents(
    "NVIDIA artificial intelligence business",
    loaded_vectorizer,
    loaded_matrix,
    loaded_chunks,
    top_k=5
)

test_results

,chunk_id,text,Similarity
22,22,Corporation. All rights reserved. 3 Table of C...,0.179330
201,201,European Union Code of Practice for General Pu...,0.176166
200,200,"experience brand or reputational harm, competi...",0.165404
23,23,address industries ranging from healthcare and...,0.049657
90,90,all. The licensed technology may not achieve t...,0.030929


In [334]:
rag_metadata = {
    "document": "NVIDIA Research Corpus",
    "chunk_count": len(rag_chunks),
    "vectorizer": "TF-IDF",
    "ngram_range": "(1,2)",
    "chunk_size_words": 150,
    "overlap_words": 30
}

In [335]:
with open(
    OUTPUT_PATH + "nvidia_rag_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        rag_metadata,
        f,
        indent=4
    )

In [336]:
print("RAG chunks:", len(rag_chunks))
print("RAG vectors:", rag_matrix.shape)
print("RAG system ready.")

RAG chunks: 649
RAG vectors: (649, 15359)
RAG system ready.


In [337]:
from scipy.sparse import load_npz

rag_matrix = load_npz(
    OUTPUT_PATH + "nvidia_rag_matrix.npz"
)

rag_chunks = pd.read_csv(
    DATA_PATH + "nvidia_rag_chunks.csv"
)

rag_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_rag_tfidf_vectorizer.pkl"
)

In [338]:
print("Chunks:", rag_chunks.shape)
print("Matrix:", rag_matrix.shape)
print("Vocabulary:", len(rag_vectorizer.vocabulary_))

Chunks: (649, 2)
Matrix: (649, 15359)
Vocabulary: 15359


In [339]:
def answer_query_with_rag(
    query,
    top_k=5
):
    results = retrieve_documents(
        query,
        rag_vectorizer,
        rag_matrix,
        rag_chunks,
        top_k=top_k
    )

    context = "\n\n".join(
        results["text"].tolist()
    )

    return {
        "query": query,
        "context": context,
        "sources": results[
            ["chunk_id", "Similarity"]
        ].to_dict("records")
    }

In [340]:
result = answer_query_with_rag(
    "What are NVIDIA's major business risks?"
)

print("Query:")
print(result["query"])

print("\nRetrieved Context:")
print(result["context"])

Query:
What are NVIDIA's major business risks?

Retrieved Context:
Related to Demand, Supply, and Manufacturing � Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand. 12 Table of Contents � Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and could harm our business. � Defects in our products have caused and could cause us to incur significant expenses to remediate and could damage our business. Risks Related to Our Global Operating Business � Adverse economic conditions may harm our business. � International sales and operations are a significant part of our business, which exposes us to risks that could harm our business. � Product, system security and data protecti

In [341]:
result = answer_query_with_rag(
    "What are NVIDIA's main products?"
)

print(result["context"])

arrangements may not deliver anticipated benefits. We also offer or plan to offer standalone software solutions, including NVIDIA AI Enterprise, NVIDIA Omniverse, NVIDIA DRIVE, and other software products. These business models or strategies may not be successful, and we may fail to sell any meaningful standalone software or services. We may incur significant costs and may not achieve any significant revenue from these offerings. Competition could adversely impact our market share and financial results. Our target markets remain competitive, and competition may intensify with expanding and changing product and service offerings, industry standards, customer and market needs, new entrants and consolidations. Our competitors� products, services and technologies, including those mentioned above in this Annual Report on Form 10-K, may be cheaper or provide better functionality or features than ours, which has resulted and may in the future result in lower-than-expected selling prices or de

In [342]:
result = answer_query_with_rag(
    "How is NVIDIA's financial performance?"
)

print(result["context"])

known warranty and indemnification issues if a loss is probable and can be reasonably estimated. Stock-based Compensation We use the closing trading price of our common stock on the date of grant, minus a dividend yield discount, as the fair value of awards of restricted stock units, or RSUs, and performance stock units, or PSUs, that are based on our corporate financial performance targets. We use a Monte Carlo simulation on the date of grant to estimate the fair value of PSUs that are based on our stock performance compared to market performance, or market-based PSUs. The compensation expense for RSUs and market-based PSUs is recognized using a straight-line attribution method over the requisite employee service period while compensation expense for PSUs is recognized using an accelerated amortization model based on performance targets probable of achievement. We estimate the fair value of shares to be issued under our employee stock

other third parties; 15 Table of Contents � the a

In [343]:
rag_test_results = pd.DataFrame([
    {
        "Query": "What are NVIDIA's major business risks?",
        "Context": answer_query_with_rag(
            "What are NVIDIA's major business risks?"
        )["context"]
    },
    {
        "Query": "What are NVIDIA's main products?",
        "Context": answer_query_with_rag(
            "What are NVIDIA's main products?"
        )["context"]
    },
    {
        "Query": "How is NVIDIA's financial performance?",
        "Context": answer_query_with_rag(
            "How is NVIDIA's financial performance?"
        )["context"]
    }
])

rag_test_results.to_csv(
    DATA_PATH + "rag_test_results.csv",
    index=False
)

In [344]:
print("RAG retrieval system ready.")
print("Chunks:", len(rag_chunks))
print("Vector dimensions:", rag_matrix.shape[1])

RAG retrieval system ready.
Chunks: 649
Vector dimensions: 15359


In [345]:
rag_chunks["text"] = (
    rag_chunks["text"]
    .astype(str)
    .str.replace("�", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [346]:
rag_matrix = rag_vectorizer.transform(
    rag_chunks["text"]
)

In [347]:
result = answer_query_with_rag(
    "What are NVIDIA's major business risks?",
    top_k=5
)

print(result["context"])

Related to Demand, Supply, and Manufacturing Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand. 12 Table of Contents Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and could harm our business. Defects in our products have caused and could cause us to incur significant expenses to remediate and could damage our business. Risks Related to Our Global Operating Business Adverse economic conditions may harm our business. International sales and operations are a significant part of our business, which exposes us to risks that could harm our business. Product, system security and data protection incidents or breaches, as well as cyber-attacks could

which could harm our 

In [348]:
def format_rag_evidence(
    query,
    top_k=5
):
    results = retrieve_documents(
        query,
        rag_vectorizer,
        rag_matrix,
        rag_chunks,
        top_k=top_k
    )

    evidence = []

    for _, row in results.iterrows():
        evidence.append({
            "chunk_id": int(row["chunk_id"]),
            "similarity": float(row["Similarity"]),
            "text": row["text"]
        })

    return evidence

In [349]:
evidence = format_rag_evidence(
    "What are NVIDIA's major business risks?"
)

for item in evidence:
    print(
        f"\nChunk {item['chunk_id']} | "
        f"Similarity: {item['similarity']:.4f}"
    )
    print(item["text"])


Chunk 82 | Similarity: 0.1224
Related to Demand, Supply, and Manufacturing Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand. 12 Table of Contents Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and could harm our business. Defects in our products have caused and could cause us to incur significant expenses to remediate and could damage our business. Risks Related to Our Global Operating Business Adverse economic conditions may harm our business. International sales and operations are a significant part of our business, which exposes us to risks that could harm our business. Product, system security and data protection incidents or breaches, as well as cyber-attac

In [350]:
rag_chunks.to_csv(
    DATA_PATH + "nvidia_rag_chunks.csv",
    index=False
)

In [351]:
save_npz(
    OUTPUT_PATH + "nvidia_rag_matrix.npz",
    rag_matrix
)

In [352]:
joblib.dump(
    rag_vectorizer,
    OUTPUT_PATH + "nvidia_rag_tfidf_vectorizer.pkl"
)

['../outputs/nvidia_rag_tfidf_vectorizer.pkl']

In [353]:
test_queries = [
    "What are NVIDIA's major business risks?",
    "What products does NVIDIA manufacture?",
    "How is NVIDIA performing financially?"
]

for query in test_queries:
    results = rag_search(
        query,
        top_k=3
    )

    print("\nQUERY:", query)

    for _, row in results.iterrows():
        print(
            f"\nSimilarity: {row['Similarity']:.4f}"
        )
        print(row["text"][:500])


QUERY: What are NVIDIA's major business risks?

Similarity: 0.1224
Related to Demand, Supply, and Manufacturing Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand. 12 Table of Contents Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and coul

Similarity: 0.1149
which could harm our business. Some of our customers have in-house expertise and internal development capabilities similar to some of ours and can use or develop their own solutions to replace those we are providing. For example, others may offer cloud-based services that compete with our AI cloud service offerings, and we may not be able to establish market share sufficient to achieve the scale necessary to

In [354]:
query_categories = {
    "research": [
        "company",
        "business",
        "product",
        "gpu",
        "technology",
        "architecture",
        "what does nvidia do",
        "nvidia products"
    ],
    "financial": [
        "revenue",
        "profit",
        "income",
        "financial",
        "margin",
        "earnings",
        "growth",
        "cash",
        "debt"
    ],
    "market": [
        "market",
        "competitor",
        "amd",
        "intel",
        "competition",
        "market share",
        "performance"
    ],
    "risk": [
        "risk",
        "regulatory",
        "export",
        "geopolitical",
        "supply chain",
        "lawsuit",
        "cyber",
        "restriction"
    ],
    "sentiment": [
        "news",
        "sentiment",
        "positive",
        "negative",
        "headline",
        "event"
    ],
    "valuation": [
        "valuation",
        "dcf",
        "pe",
        "p/e",
        "price",
        "value",
        "worth",
        "enterprise value"
    ],
    "stock": [
        "stock",
        "share",
        "return",
        "price movement",
        "up",
        "down",
        "prediction"
    ]
}

In [355]:
def classify_query(query):
    query_lower = query.lower()

    category_scores = {}

    for category, keywords in query_categories.items():
        score = 0

        for keyword in keywords:
            if keyword in query_lower:
                score += 1

        category_scores[category] = score

    priority = [
        "stock",
        "financial",
        "valuation",
        "risk",
        "market",
        "sentiment",
        "research"
    ]

    best_category = "research"
    best_score = 0

    for category in priority:
        score = category_scores.get(category, 0)

        if score > best_score:
            best_score = score
            best_category = category

    return best_category, category_scores

In [356]:
test_queries = [
    "If NVIDIA stock is falling should I buy or sell?",
    "How is NVIDIA financially?",
    "What are NVIDIA's biggest risks?",
    "How does NVIDIA compare with AMD?",
    "What is NVIDIA's valuation?",
    "What does NVIDIA manufacture?",
    "What is the sentiment around NVIDIA news?"
]

for query in test_queries:
    category, scores = classify_query(query)

    print(
        f"\nQuery: {query}"
    )
    print(
        f"Category: {category}"
    )


Query: If NVIDIA stock is falling should I buy or sell?
Category: stock

Query: How is NVIDIA financially?
Category: financial

Query: What are NVIDIA's biggest risks?
Category: risk

Query: How does NVIDIA compare with AMD?
Category: market

Query: What is NVIDIA's valuation?
Category: valuation

Query: What does NVIDIA manufacture?
Category: research

Query: What is the sentiment around NVIDIA news?
Category: sentiment


In [357]:
def process_query(query, top_k=5):
    category, scores = classify_query(query)

    rag_results = retrieve_documents(
        query,
        rag_vectorizer,
        rag_matrix,
        rag_chunks,
        top_k=top_k
    )

    return {
        "query": query,
        "category": category,
        "category_scores": scores,
        "retrieved_documents": rag_results
    }

In [358]:
query_result = process_query(
    "What are NVIDIA's major business risks?"
)

print("Query:", query_result["query"])
print("Category:", query_result["category"])

query_result["retrieved_documents"]

Query: What are NVIDIA's major business risks?
Category: risk


,chunk_id,text,Similarity
82,82,"Related to Demand, Supply, and Manufacturing L...",0.122362
93,93,which could harm our business. Some of our cus...,0.114869
594,594,"our legal proceedings since January 25, 2026. ...",0.069905
136,136,developments in or around any of the major cou...,0.068848
135,135,and other natural or man-made disasters and ca...,0.067656


In [359]:
query_test = {
    "Query": "What are NVIDIA's major business risks?",
    "Category": query_result["category"]
}

with open(
    OUTPUT_PATH + "query_routing_test.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        query_test,
        f,
        indent=4
    )

In [360]:
joblib.dump(
    query_categories,
    OUTPUT_PATH + "nvidia_query_categories.pkl"
)

['../outputs/nvidia_query_categories.pkl']

In [361]:
stock_model = joblib.load(
    OUTPUT_PATH + "nvidia_logistic_final_model.pkl"
)

stock_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_logistic_scaler.pkl"
)

sentiment_model = joblib.load(
    OUTPUT_PATH + "nvidia_financial_sentiment_model.pkl"
)

sentiment_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_sentiment_tfidf_vectorizer.pkl"
)

risk_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_risk_model.pkl"
)

risk_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_risk_tfidf_vectorizer.pkl"
)

market_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_market_competitor_model.pkl"
)

market_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_market_competitor_scaler.pkl"
)

research_model = joblib.load(
    OUTPUT_PATH + "nvidia_research_topic_model.pkl"
)

research_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_research_tfidf_vectorizer.pkl"
)

In [362]:
def route_to_model(query):
    category, scores = classify_query(query)

    result = {
        "query": query,
        "category": category,
        "scores": scores
    }

    if category == "research":
        result["model"] = "Research Model"

    elif category == "financial":
        result["model"] = "Financial Analysis Model"

    elif category == "market":
        result["model"] = "Market & Competitor Model"

    elif category == "risk":
        result["model"] = "Risk Classification Model"

    elif category == "sentiment":
        result["model"] = "News Sentiment Model"

    elif category == "valuation":
        result["model"] = "Valuation Model"

    elif category == "stock":
        result["model"] = "Stock Prediction Model"

    return result

In [363]:
queries = [
    "What are NVIDIA's major risks?",
    "How is NVIDIA's financial performance?",
    "Who are NVIDIA's competitors?",
    "What is NVIDIA's valuation?",
    "Will NVIDIA stock go up?"
]

for query in queries:
    result = route_to_model(query)

    print(
        f"Query: {query}\n"
        f"Category: {result['category']}\n"
        f"Model: {result['model']}\n"
    )

Query: What are NVIDIA's major risks?
Category: risk
Model: Risk Classification Model

Query: How is NVIDIA's financial performance?
Category: financial
Model: Financial Analysis Model

Query: Who are NVIDIA's competitors?
Category: valuation
Model: Valuation Model

Query: What is NVIDIA's valuation?
Category: valuation
Model: Valuation Model

Query: Will NVIDIA stock go up?
Category: stock
Model: Stock Prediction Model



In [364]:
def research_prediction(query):
    vector = research_vectorizer.transform([query])
    topic = research_model.predict(vector)[0]

    return {
        "topic": int(topic)
    }

In [365]:
def sentiment_prediction(query):
    vector = sentiment_vectorizer.transform([query])
    prediction = sentiment_model.predict(vector)[0]
    probability = sentiment_model.predict_proba(vector).max()

    return {
        "sentiment": prediction,
        "confidence": float(probability)
    }

In [366]:
def risk_prediction(query):
    vector = risk_vectorizer.transform([query])
    prediction = risk_model.predict(vector)[0]

    confidence = None

    if hasattr(risk_model, "predict_proba"):
        confidence = float(
            risk_model.predict_proba(vector).max()
        )

    return {
        "high_risk": int(prediction),
        "confidence": confidence
    }

In [367]:
def process_user_query(query, top_k=5):
    routed = route_to_model(query)

    rag_results = retrieve_documents(
        query,
        rag_vectorizer,
        rag_matrix,
        rag_chunks,
        top_k=top_k
    )

    return {
        "query": query,
        "category": routed["category"],
        "model": routed["model"],
        "rag_results": rag_results
    }

In [368]:
result = process_user_query(
    "What are NVIDIA's major business risks?"
)

print("Query:", result["query"])
print("Category:", result["category"])
print("Model:", result["model"])
print("\nRetrieved Evidence:")
print(result["rag_results"][["chunk_id", "Similarity"]])

Query: What are NVIDIA's major business risks?
Category: risk
Model: Risk Classification Model

Retrieved Evidence:
     chunk_id  Similarity
82         82    0.122362
93         93    0.114869
594       594    0.069905
136       136    0.068848
135       135    0.067656


In [369]:
query_log = pd.DataFrame({
    "Query": [result["query"]],
    "Category": [result["category"]],
    "Model": [result["model"]],
    "Retrieved_Chunks": [
        len(result["rag_results"])
    ]
})

query_log.to_csv(
    DATA_PATH + "query_log.csv",
    index=False
)

In [370]:
def generate_model_output(query):

    routed = route_to_model(query)
    category = routed["category"]

    if category == "stock":
        output = stock_model_prediction()

    elif category == "financial":
        output = financial_model_prediction()

    elif category == "market":
        output = market_model_prediction()

    elif category == "risk":
        output = risk_prediction(query)

    elif category == "sentiment":
        output = sentiment_prediction(query)

    elif category == "research":
        output = research_prediction(query)

    elif category == "valuation":
        output = valuation_model_prediction()

    else:
        output = {
            "status": "No model available"
        }

    return {
        "query": query,
        "category": category,
        "model": routed["model"],
        "output": output
    }

In [371]:
result = generate_model_output(
    "What are NVIDIA's major business risks?"
)

result

{'query': "What are NVIDIA's major business risks?",
 'category': 'risk',
 'model': 'Risk Classification Model',
 'output': {'high_risk': 0, 'confidence': 0.6427039538487455}}

In [372]:
def rag_model_pipeline(
    query,
    top_k=5
):
    routed = route_to_model(query)

    rag_results = retrieve_documents(
        query,
        rag_vectorizer,
        rag_matrix,
        rag_chunks,
        top_k=top_k
    )

    model_result = generate_model_output(
        query
    )

    return {
        "query": query,
        "category": routed["category"],
        "model": routed["model"],
        "model_result": model_result["output"],
        "evidence": rag_results[
            [
                "chunk_id",
                "Similarity",
                "text"
            ]
        ]
    }

In [373]:
result = rag_model_pipeline(
    "What are NVIDIA's major business risks?"
)

print("Query:", result["query"])
print("Category:", result["category"])
print("Model:", result["model"])

print("\nModel Result:")
print(result["model_result"])

print("\nEvidence:")
print(result["evidence"])

Query: What are NVIDIA's major business risks?
Category: risk
Model: Risk Classification Model

Model Result:
{'high_risk': 0, 'confidence': 0.6427039538487455}

Evidence:
     chunk_id  Similarity                                               text
82         82    0.122362  Related to Demand, Supply, and Manufacturing L...
93         93    0.114869  which could harm our business. Some of our cus...
594       594    0.069905  our legal proceedings since January 25, 2026. ...
136       136    0.068848  developments in or around any of the major cou...
135       135    0.067656  and other natural or man-made disasters and ca...


In [374]:
def create_structured_response(query):
    result = rag_model_pipeline(query)

    response = {
        "query": result["query"],
        "category": result["category"],
        "model": result["model"],
        "analysis": result["model_result"],
        "evidence": result["evidence"].to_dict(
            orient="records"
        )
    }

    return response

In [375]:
response = create_structured_response(
    "What are NVIDIA's major business risks?"
)

response

{'query': "What are NVIDIA's major business risks?",
 'category': 'risk',
 'model': 'Risk Classification Model',
 'analysis': {'high_risk': 0, 'confidence': 0.6427039538487455},
 'evidence': [{'chunk_id': 82,
   'Similarity': 0.1223615300102363,
   'text': 'Related to Demand, Supply, and Manufacturing Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand. 12 Table of Contents Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and could harm our business. Defects in our products have caused and could cause us to incur significant expenses to remediate and could damage our business. Risks Related to Our Global Operating Business Adverse economic conditions may harm our busi

In [376]:
with open(
    OUTPUT_PATH + "sample_nvidia_response.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        response,
        f,
        indent=4,
        default=str
    )

In [377]:
user_query = input(
    "Ask a question about NVIDIA: "
)

response = create_structured_response(
    user_query
)

print("\nCategory:", response["category"])
print("Model:", response["model"])
print("\nAnalysis:")
print(response["analysis"])

print("\nEvidence:")
for evidence in response["evidence"]:
    print(
        f"\nSimilarity: {evidence['Similarity']:.4f}"
    )
    print(evidence["text"][:700])

Ask a question about NVIDIA:  Hi



Category: research
Model: Research Model

Analysis:
{'topic': 0}

Evidence:

Similarity: 0.0000
August 26, 2026 NVIDIA Corporation By: /s/ Colette M. Kress Colette M. Kress Executive Vice President and Chief Financial Officer (Duly Authorized Officer and Principal Financial Officer) 43

Similarity: 0.0000
harmed if we are unsuccessful or if we are prohibited from making or selling our products. From time to time, we are involved in lawsuits or other legal proceedings alleging patent infringement or other IP rights violations by us, our employees or parties that we have agreed to indemnify. An unfavorable ruling could include significant damages, invalidation of one or more patents, indemnification of third parties, payment of lost profits, or injunctive relief. Claims that our products or processes infringe the IP rights of others, regardless of their merit, could cause us to incur significant costs to respond to, defend, and resolve such claims, and they may also divert the efforts a

In [378]:
def generate_text_response(query, result):
    category = result["category"]
    model = result["model"]
    analysis = result["analysis"]
    evidence = result["evidence"]

    response = []

    response.append(f"Question: {query}")
    response.append(f"Category: {category}")
    response.append(f"Analysis Model: {model}")

    response.append("\nAnalysis:")
    response.append(str(analysis))

    response.append("\nSupporting Evidence:")

    for i, item in enumerate(evidence[:3], start=1):
        response.append(
            f"\n{i}. {item['text'][:1000]}"
        )
        response.append(
            f"Similarity: {item['Similarity']:.4f}"
        )

    return "\n".join(response)

In [379]:
query = "What are NVIDIA's major business risks?"

response_data = create_structured_response(
    query
)

final_response = generate_text_response(
    query,
    response_data
)

print(final_response)

Question: What are NVIDIA's major business risks?
Category: risk
Analysis Model: Risk Classification Model

Analysis:
{'high_risk': 0, 'confidence': 0.6427039538487455}

Supporting Evidence:

1. Related to Demand, Supply, and Manufacturing Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand. 12 Table of Contents Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and could harm our business. Defects in our products have caused and could cause us to incur significant expenses to remediate and could damage our business. Risks Related to Our Global Operating Business Adverse economic conditions may harm our business. International sales and operations are a significant part

In [380]:
test_questions = [
    "What are NVIDIA's major business risks?",
    "What does NVIDIA do?",
    "How is NVIDIA performing financially?",
    "Who are NVIDIA's main competitors?"
]

for question in test_questions:
    result = create_structured_response(question)

    print("\n" + "=" * 80)
    print(generate_text_response(question, result))


Question: What are NVIDIA's major business risks?
Category: risk
Analysis Model: Risk Classification Model

Analysis:
{'high_risk': 0, 'confidence': 0.6427039538487453}

Supporting Evidence:

1. Related to Demand, Supply, and Manufacturing Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand. 12 Table of Contents Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and could harm our business. Defects in our products have caused and could cause us to incur significant expenses to remediate and could damage our business. Risks Related to Our Global Operating Business Adverse economic conditions may harm our business. International sales and operations are a significant par

In [381]:
with open(
    OUTPUT_PATH + "nvidia_query_system_ready.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        "NVIDIA Natural Language Query System Ready"
    )

In [382]:
user_query = input(
    "Ask anything about NVIDIA: "
)

result = create_structured_response(
    user_query
)

final_response = generate_text_response(
    user_query,
    result
)

print("\n" + final_response)

Ask anything about NVIDIA:  Hi



Question: Hi
Category: research
Analysis Model: Research Model

Analysis:
{'topic': 0}

Supporting Evidence:

1. August 26, 2026 NVIDIA Corporation By: /s/ Colette M. Kress Colette M. Kress Executive Vice President and Chief Financial Officer (Duly Authorized Officer and Principal Financial Officer) 43
Similarity: 0.0000

2. harmed if we are unsuccessful or if we are prohibited from making or selling our products. From time to time, we are involved in lawsuits or other legal proceedings alleging patent infringement or other IP rights violations by us, our employees or parties that we have agreed to indemnify. An unfavorable ruling could include significant damages, invalidation of one or more patents, indemnification of third parties, payment of lost profits, or injunctive relief. Claims that our products or processes infringe the IP rights of others, regardless of their merit, could cause us to incur significant costs to respond to, defend, and resolve such claims, and they may also 

In [383]:
def stock_model_prediction():
    latest_row = stock_model_df[
        model_features
    ].iloc[[-1]]

    latest_scaled = stock_scaler.transform(
        latest_row
    )

    prediction = stock_model.predict(
        latest_scaled
    )[0]

    probability = stock_model.predict_proba(
        latest_scaled
    )[0]

    return {
        "prediction": "UP" if prediction == 1 else "DOWN",
        "probability_up": float(probability[1]),
        "probability_down": float(probability[0]),
        "confidence": float(probability.max())
    }

In [384]:
def financial_model_prediction():
    latest_row = financial_model_df[
        [
            "revenue",
            "net_income",
            "Revenue_Growth",
            "Net_Income_Growth",
            "profit_margin"
        ]
    ].iloc[[-1]]

    latest_scaled = financial_scaler.transform(
        latest_row
    )

    prediction = best_financial_model.predict(
        latest_scaled
    )[0]

    return {
        "financial_health": (
            "STRONG" if prediction == 1 else "WEAK"
        )
    }

In [385]:
def market_model_prediction():
    latest_row = market_model_df[
        market_features
    ].iloc[[-1]]

    latest_scaled = market_scaler.transform(
        latest_row
    )

    prediction = best_market_model.predict(
        latest_scaled
    )[0]

    return {
        "market_position": (
            "OUTPERFORMING"
            if prediction == 1
            else "NOT OUTPERFORMING"
        )
    }

In [386]:
def risk_model_prediction(query):
    vector = risk_vectorizer.transform(
        [query]
    )

    prediction = risk_model.predict(
        vector
    )[0]

    confidence = None

    if hasattr(risk_model, "predict_proba"):
        confidence = float(
            risk_model.predict_proba(vector).max()
        )

    return {
        "high_risk": int(prediction),
        "confidence": confidence
    }

In [387]:
def generate_model_output(query):
    routed = route_to_model(query)
    category = routed["category"]

    if category == "stock":
        output = stock_model_prediction()

    elif category == "financial":
        output = financial_model_prediction()

    elif category == "market":
        output = market_model_prediction()

    elif category == "risk":
        output = risk_model_prediction(query)

    elif category == "sentiment":
        output = sentiment_prediction(query)

    elif category == "research":
        output = research_prediction(query)

    elif category == "valuation":
        output = {
            "dcf_enterprise_value": float(
                enterprise_value
            )
        }

    else:
        output = {
            "status": "No model available"
        }

    return {
        "query": query,
        "category": category,
        "model": routed["model"],
        "output": output
    }

In [388]:
stock_model_df = pd.read_csv(
    DATA_PATH + "nvidia_stock_model_data.csv"
)

In [389]:
model_features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Daily_Return",
    "MA_20",
    "MA_50",
    "MA_200",
    "Volatility_20",
    "Momentum_10",
    "Momentum_20",
    "Price_Change_Pct",
    "High_Low_Range",
    "Volume_Change",
    "Volume_MA_20"
]

In [390]:
result = generate_model_output(
    query
)

result

{'query': "What are NVIDIA's major business risks?",
 'category': 'risk',
 'model': 'Risk Classification Model',
 'output': {'high_risk': 0, 'confidence': 0.6427039538487455}}

In [391]:
result = rag_model_pipeline(
    query
)

print("Category:", result["category"])
print("Model:", result["model"])
print("Model Result:", result["model_result"])

Category: risk
Model: Risk Classification Model
Model Result: {'high_risk': 0, 'confidence': 0.6427039538487455}


In [392]:
financial_model_df = pd.read_csv(
    DATA_PATH + "nvidia_financial_model_data.csv"
)

In [393]:
financial_features = [
    "revenue",
    "net_income",
    "Revenue_Growth",
    "Net_Income_Growth",
    "profit_margin"
]

In [394]:
def financial_model_prediction():
    latest_row = financial_model_df[
        financial_features
    ].iloc[[-1]]

    latest_scaled = financial_scaler.transform(
        latest_row
    )

    prediction = best_financial_model.predict(
        latest_scaled
    )[0]

    return {
        "financial_health": (
            "STRONG"
            if prediction == 1
            else "WEAK"
        )
    }

In [395]:
market_model_df = pd.read_csv(
    DATA_PATH + "nvidia_competitor_comparison.csv"
)

In [396]:
market_features = [
    "NVIDIA_Return",
    "AMD_Return",
    "Intel_Return",
    "NVIDIA_vs_AMD",
    "NVIDIA_vs_Intel",
    "NVIDIA_vs_Peer_Average"
]

In [397]:
def market_model_prediction():
    latest_row = market_model_df[
        market_features
    ].iloc[[-1]]

    latest_scaled = market_scaler.transform(
        latest_row
    )

    prediction = best_market_model.predict(
        latest_scaled
    )[0]

    return {
        "market_position": (
            "OUTPERFORMING"
            if prediction == 1
            else "NOT OUTPERFORMING"
        )
    }

In [398]:
best_financial_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_financial_model.pkl"
)

financial_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_financial_scaler.pkl"
)

In [399]:
financial_model_prediction()

{'financial_health': 'STRONG'}

In [400]:
market_model_df["NVIDIA_vs_Peer_Average"] = (
    market_model_df["NVIDIA_Return"]
    - (
        market_model_df["AMD_Return"]
        + market_model_df["Intel_Return"]
    ) / 2
)

In [401]:
market_model_df[market_features].tail()

,NVIDIA_Return,AMD_Return,Intel_Return,NVIDIA_vs_AMD,NVIDIA_vs_Intel,NVIDIA_vs_Peer_Average
6063,-3.218110,0.625500,0.245497,-3.843610,-3.463607,-3.653608
6064,-4.177522,2.009395,1.510208,-6.186918,-5.687730,-5.937324
6065,0.661663,-2.416215,-3.297151,3.077877,3.958814,3.518346
6066,-1.153960,-1.074641,-1.663200,-0.079319,0.509240,0.214961
6067,2.150143,0.689949,1.691330,1.460194,0.458813,0.959503


In [402]:
best_market_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_market_competitor_model.pkl"
)

market_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_market_competitor_scaler.pkl"
)

print("Market model loaded")
print(type(best_market_model).__name__)

Market model loaded
LogisticRegression


In [403]:
market_features = [
    "NVIDIA_Return",
    "AMD_Return",
    "Intel_Return",
    "NVIDIA_vs_AMD",
    "NVIDIA_vs_Intel",
    "NVIDIA_vs_Peer_Average"
]

market_model_df["NVIDIA_vs_AMD"] = (
    market_model_df["NVIDIA_Return"]
    - market_model_df["AMD_Return"]
)

market_model_df["NVIDIA_vs_Intel"] = (
    market_model_df["NVIDIA_Return"]
    - market_model_df["Intel_Return"]
)

market_model_df["NVIDIA_vs_Peer_Average"] = (
    market_model_df["NVIDIA_Return"]
    - (
        market_model_df["AMD_Return"]
        + market_model_df["Intel_Return"]
    ) / 2
)

print(market_model_df[market_features].tail())

      NVIDIA_Return  AMD_Return  Intel_Return  NVIDIA_vs_AMD  NVIDIA_vs_Intel  \
6063      -3.218110    0.625500      0.245497      -3.843610        -3.463607   
6064      -4.177522    2.009395      1.510208      -6.186918        -5.687730   
6065       0.661663   -2.416215     -3.297151       3.077877         3.958814   
6066      -1.153960   -1.074641     -1.663200      -0.079319         0.509240   
6067       2.150143    0.689949      1.691330       1.460194         0.458813   

      NVIDIA_vs_Peer_Average  
6063               -3.653608  
6064               -5.937324  
6065                3.518346  
6066                0.214961  
6067                0.959503  


In [404]:
def market_model_prediction():

    latest_row = market_model_df[
        market_features
    ].iloc[[-1]]

    latest_scaled = market_scaler.transform(
        latest_row
    )

    prediction = best_market_model.predict(
        latest_scaled
    )[0]

    return {
        "market_position": (
            "OUTPERFORMING"
            if prediction == 1
            else "NOT OUTPERFORMING"
        )
    }

In [405]:
market_model_prediction()

{'market_position': 'OUTPERFORMING'}

In [406]:
def valuation_model_prediction():

    try:
        valuation_data = pd.read_csv(
            DATA_PATH + "nvidia_valuation_data.csv"
        )

        latest = valuation_data.iloc[-1]

        result = {}

        for col in valuation_data.columns:
            if any(term in col.lower() for term in [
                "enterprise",
                "valuation",
                "market_cap",
                "pe",
                "price"
            ]):
                try:
                    result[col] = float(latest[col])
                except:
                    pass

        if not result:
            return {
                "status": "Valuation data loaded",
                "columns": list(valuation_data.columns)
            }

        return result

    except Exception as e:
        return {
            "status": "Valuation data unavailable",
            "error": str(e)
        }

In [407]:
valuation_model_prediction()

{'status': 'Valuation data loaded',
 'columns': ['fy',
  'revenue',
  'net_income',
  'Revenue_Growth',
  'Net_Income_Growth',
  'profit_margin']}

In [408]:
synthesis_data = pd.DataFrame([
    {
        "stock_signal": 1,
        "financial_signal": 1,
        "market_signal": 1,
        "risk_signal": 0,
        "valuation_signal": 1,
        "sentiment_signal": 1,
        "recommendation": "BUY"
    },
    {
        "stock_signal": 0,
        "financial_signal": 1,
        "market_signal": 1,
        "risk_signal": 0,
        "valuation_signal": 1,
        "sentiment_signal": 1,
        "recommendation": "HOLD"
    },
    {
        "stock_signal": 0,
        "financial_signal": 0,
        "market_signal": 0,
        "risk_signal": 1,
        "valuation_signal": 0,
        "sentiment_signal": 0,
        "recommendation": "SELL"
    },
    {
        "stock_signal": 1,
        "financial_signal": 1,
        "market_signal": 0,
        "risk_signal": 1,
        "valuation_signal": 0,
        "sentiment_signal": 1,
        "recommendation": "HOLD"
    },
    {
        "stock_signal": 1,
        "financial_signal": 1,
        "market_signal": 1,
        "risk_signal": 0,
        "valuation_signal": 0,
        "sentiment_signal": 1,
        "recommendation": "BUY"
    },
    {
        "stock_signal": 0,
        "financial_signal": 1,
        "market_signal": 0,
        "risk_signal": 1,
        "valuation_signal": 0,
        "sentiment_signal": 0,
        "recommendation": "SELL"
    }
])

print(synthesis_data)

   stock_signal  financial_signal  market_signal  risk_signal  \
0             1                 1              1            0   
1             0                 1              1            0   
2             0                 0              0            1   
3             1                 1              0            1   
4             1                 1              1            0   
5             0                 1              0            1   

   valuation_signal  sentiment_signal recommendation  
0                 1                 1            BUY  
1                 1                 1           HOLD  
2                 0                 0           SELL  
3                 0                 1           HOLD  
4                 0                 1            BUY  
5                 0                 0           SELL  


In [409]:
from sklearn.preprocessing import LabelEncoder

synthesis_encoder = LabelEncoder()

synthesis_data["recommendation_encoded"] = (
    synthesis_encoder.fit_transform(
        synthesis_data["recommendation"]
    )
)

print(
    synthesis_data[
        ["recommendation", "recommendation_encoded"]
    ]
)

  recommendation  recommendation_encoded
0            BUY                       0
1           HOLD                       1
2           SELL                       2
3           HOLD                       1
4            BUY                       0
5           SELL                       2


In [410]:
from sklearn.ensemble import RandomForestClassifier

synthesis_features = [
    "stock_signal",
    "financial_signal",
    "market_signal",
    "risk_signal",
    "valuation_signal",
    "sentiment_signal"
]

X_synthesis = synthesis_data[synthesis_features]
y_synthesis = synthesis_data["recommendation_encoded"]

investment_synthesis_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

investment_synthesis_model.fit(
    X_synthesis,
    y_synthesis
)

print("Investment Synthesis Model trained")

Investment Synthesis Model trained


In [411]:
joblib.dump(
    investment_synthesis_model,
    OUTPUT_PATH + "nvidia_investment_synthesis_model.pkl"
)

joblib.dump(
    synthesis_encoder,
    OUTPUT_PATH + "nvidia_investment_synthesis_encoder.pkl"
)

print("Investment Synthesis Model saved")

Investment Synthesis Model saved


In [412]:
investment_synthesis_model = joblib.load(
    OUTPUT_PATH + "nvidia_investment_synthesis_model.pkl"
)

synthesis_encoder = joblib.load(
    OUTPUT_PATH + "nvidia_investment_synthesis_encoder.pkl"
)

def investment_synthesis_prediction(
    stock_result,
    financial_result,
    market_result,
    risk_result,
    valuation_result,
    sentiment_result
):

    stock_signal = (
        1
        if stock_result.get("prediction") == "UP"
        else 0
    )

    financial_signal = (
        1
        if financial_result.get("financial_health") == "STRONG"
        else 0
    )

    market_signal = (
        1
        if market_result.get("market_position") == "OUTPERFORMING"
        else 0
    )

    risk_signal = risk_result.get(
        "high_risk",
        0
    )

    valuation_signal = 1

    sentiment_signal = (
        1
        if sentiment_result.get("sentiment") == "positive"
        else 0
    )

    input_data = pd.DataFrame([{
        "stock_signal": stock_signal,
        "financial_signal": financial_signal,
        "market_signal": market_signal,
        "risk_signal": risk_signal,
        "valuation_signal": valuation_signal,
        "sentiment_signal": sentiment_signal
    }])

    prediction = investment_synthesis_model.predict(
        input_data
    )[0]

    recommendation = synthesis_encoder.inverse_transform(
        [prediction]
    )[0]

    return {
        "recommendation": recommendation,
        "signals": input_data.iloc[0].to_dict()
    }

In [413]:
def collect_current_model_signals(query):

    stock_result = stock_model_prediction()
    financial_result = financial_model_prediction()
    market_result = market_model_prediction()
    risk_result = risk_model_prediction(query)
    sentiment_result = sentiment_prediction(query)
    valuation_result = valuation_model_prediction()

    signals = {
        "stock_signal": 1 if stock_result.get("prediction") == "UP" else 0,
        "financial_signal": 1 if financial_result.get("financial_health") == "STRONG" else 0,
        "market_signal": 1 if market_result.get("market_position") == "OUTPERFORMING" else 0,
        "risk_signal": int(risk_result.get("high_risk", 0)),
        "sentiment_signal": 1 if sentiment_result.get("sentiment") == "positive" else 0,
        "valuation_result": valuation_result
    }

    return {
        "stock": stock_result,
        "financial": financial_result,
        "market": market_result,
        "risk": risk_result,
        "sentiment": sentiment_result,
        "valuation": valuation_result,
        "signals": signals
    }

In [414]:
def sentiment_prediction(query):

    try:
        vector = sentiment_vectorizer.transform([query])

        prediction = sentiment_model.predict(vector)[0]

        probability = sentiment_model.predict_proba(vector).max()

        return {
            "sentiment": prediction,
            "confidence": float(probability),
            "status": "model"
        }

    except Exception:
        return {
            "sentiment": "neutral",
            "confidence": 0.0,
            "status": "fallback"
        }

In [415]:
current_signals = collect_current_model_signals(
    "Should NVIDIA be considered attractive for investment?"
)

current_signals

{'stock': {'prediction': 'UP',
  'probability_up': 0.9982780854862398,
  'probability_down': 0.0017219145137602476,
  'confidence': 0.9982780854862398},
 'financial': {'financial_health': 'STRONG'},
 'market': {'market_position': 'OUTPERFORMING'},
 'risk': {'high_risk': 0, 'confidence': 0.7254620163652168},
 'sentiment': {'sentiment': 'neutral',
  'confidence': 0.0,
  'status': 'fallback'},
 'valuation': {'status': 'Valuation data loaded',
  'columns': ['fy',
   'revenue',
   'net_income',
   'Revenue_Growth',
   'Net_Income_Growth',
   'profit_margin']},
 'signals': {'stock_signal': 1,
  'financial_signal': 1,
  'market_signal': 1,
  'risk_signal': 0,
  'sentiment_signal': 0,
  'valuation_result': {'status': 'Valuation data loaded',
   'columns': ['fy',
    'revenue',
    'net_income',
    'Revenue_Growth',
    'Net_Income_Growth',
    'profit_margin']}}}

In [416]:
def collect_current_model_signals(query):

    stock_result = stock_model_prediction()
    financial_result = financial_model_prediction()
    market_result = market_model_prediction()
    risk_result = risk_model_prediction(query)
    sentiment_result = sentiment_prediction(query)
    valuation_result = valuation_model_prediction()

    signals = {
        "stock_signal": (
            1
            if stock_result.get("prediction") == "UP"
            else 0
        ),

        "financial_signal": (
            1
            if financial_result.get("financial_health") == "STRONG"
            else 0
        ),

        "market_signal": (
            1
            if market_result.get("market_position") == "OUTPERFORMING"
            else 0
        ),

        "risk_signal": int(
            risk_result.get("high_risk", 0)
        ),

        "sentiment_signal": (
            1
            if sentiment_result.get("sentiment") == "positive"
            else 0
        )
    }

    return {
        "stock": stock_result,
        "financial": financial_result,
        "market": market_result,
        "risk": risk_result,
        "sentiment": sentiment_result,
        "valuation": valuation_result,
        "signals": signals
    }

In [417]:
current_signals = collect_current_model_signals(
    "Should NVIDIA be considered attractive for investment?"
)

print("Signals:")
print(current_signals["signals"])

print("\nSentiment:")
print(current_signals["sentiment"])

print("\nValuation:")
print(current_signals["valuation"])

Signals:
{'stock_signal': 1, 'financial_signal': 1, 'market_signal': 1, 'risk_signal': 0, 'sentiment_signal': 0}

Sentiment:
{'sentiment': 'neutral', 'confidence': 0.0, 'status': 'fallback'}

Valuation:
{'status': 'Valuation data loaded', 'columns': ['fy', 'revenue', 'net_income', 'Revenue_Growth', 'Net_Income_Growth', 'profit_margin']}


In [418]:
from itertools import product

synthesis_rows = []

for stock_signal, financial_signal, market_signal, risk_signal, sentiment_signal in product(
    [0, 1],
    repeat=5
):

    positive_score = (
        stock_signal
        + financial_signal
        + market_signal
        + sentiment_signal
    )

    negative_score = risk_signal

    total_score = positive_score - negative_score

    if total_score >= 3:
        recommendation = "BUY"

    elif total_score <= 0:
        recommendation = "SELL"

    else:
        recommendation = "HOLD"

    synthesis_rows.append({
        "stock_signal": stock_signal,
        "financial_signal": financial_signal,
        "market_signal": market_signal,
        "risk_signal": risk_signal,
        "sentiment_signal": sentiment_signal,
        "recommendation": recommendation
    })

synthesis_data = pd.DataFrame(
    synthesis_rows
)

print("Training rows:", len(synthesis_data))
print(synthesis_data.head())

Training rows: 32
   stock_signal  financial_signal  market_signal  risk_signal  \
0             0                 0              0            0   
1             0                 0              0            0   
2             0                 0              0            1   
3             0                 0              0            1   
4             0                 0              1            0   

   sentiment_signal recommendation  
0                 0           SELL  
1                 1           HOLD  
2                 0           SELL  
3                 1           SELL  
4                 0           HOLD  


In [419]:
synthesis_encoder = LabelEncoder()

synthesis_data["recommendation_encoded"] = (
    synthesis_encoder.fit_transform(
        synthesis_data["recommendation"]
    )
)

print(
    synthesis_data[
        ["recommendation", "recommendation_encoded"]
    ].drop_duplicates()
)

   recommendation  recommendation_encoded
0            SELL                       2
1            HOLD                       1
13            BUY                       0


In [420]:
synthesis_features = [
    "stock_signal",
    "financial_signal",
    "market_signal",
    "risk_signal",
    "sentiment_signal"
]

X_synthesis = synthesis_data[
    synthesis_features
]

y_synthesis = synthesis_data[
    "recommendation_encoded"
]

investment_synthesis_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

investment_synthesis_model.fit(
    X_synthesis,
    y_synthesis
)

print("Investment Synthesis Model trained")

Investment Synthesis Model trained


In [421]:
from sklearn.metrics import accuracy_score

In [422]:
from sklearn.metrics import accuracy_score, classification_report

synthesis_predictions = (
    investment_synthesis_model.predict(
        X_synthesis
    )
)

print(
    "Accuracy:",
    accuracy_score(
        y_synthesis,
        synthesis_predictions
    )
)

print(
    classification_report(
        y_synthesis,
        synthesis_predictions,
        target_names=synthesis_encoder.classes_
    )
)

Accuracy: 1.0
              precision    recall  f1-score   support

         BUY       1.00      1.00      1.00         6
        HOLD       1.00      1.00      1.00        20
        SELL       1.00      1.00      1.00         6

    accuracy                           1.00        32
   macro avg       1.00      1.00      1.00        32
weighted avg       1.00      1.00      1.00        32



In [423]:
def investment_synthesis_prediction(query):

    results = collect_current_model_signals(query)

    signals = results["signals"]

    input_data = pd.DataFrame([{
        "stock_signal": signals["stock_signal"],
        "financial_signal": signals["financial_signal"],
        "market_signal": signals["market_signal"],
        "risk_signal": signals["risk_signal"],
        "sentiment_signal": signals["sentiment_signal"]
    }])

    prediction = investment_synthesis_model.predict(
        input_data
    )[0]

    recommendation = synthesis_encoder.inverse_transform(
        [prediction]
    )[0]

    probabilities = {}

    if hasattr(
        investment_synthesis_model,
        "predict_proba"
    ):
        probability_values = (
            investment_synthesis_model.predict_proba(
                input_data
            )[0]
        )

        for label, probability in zip(
            synthesis_encoder.classes_,
            probability_values
        ):
            probabilities[label] = float(
                probability
            )

    confidence = (
        float(max(probabilities.values()))
        if probabilities
        else None
    )

    return {
        "recommendation": recommendation,
        "confidence": confidence,
        "probabilities": probabilities,
        "signals": signals,
        "model_outputs": {
            "stock": results["stock"],
            "financial": results["financial"],
            "market": results["market"],
            "risk": results["risk"],
            "sentiment": results["sentiment"],
            "valuation": results["valuation"]
        }
    }

In [424]:
synthesis_result = investment_synthesis_prediction(
    "Should I invest in NVIDIA?"
)

print(
    "Recommendation:",
    synthesis_result["recommendation"]
)

print(
    "Confidence:",
    synthesis_result["confidence"]
)

print(
    "Probabilities:",
    synthesis_result["probabilities"]
)

print(
    "Signals:",
    synthesis_result["signals"]
)

Recommendation: BUY
Confidence: 0.71
Probabilities: {'BUY': 0.71, 'HOLD': 0.29, 'SELL': 0.0}
Signals: {'stock_signal': 1, 'financial_signal': 1, 'market_signal': 1, 'risk_signal': 0, 'sentiment_signal': 0}


In [425]:
with open(
    OUTPUT_PATH + "nvidia_investment_synthesis_result.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        synthesis_result,
        f,
        indent=4,
        default=str
    )

print("Synthesis result saved")

Synthesis result saved


In [426]:
def red_team_verification(synthesis_result):

    recommendation = synthesis_result["recommendation"]
    signals = synthesis_result["signals"]

    contradictions = []

    if recommendation == "BUY":

        if signals["stock_signal"] == 0:
            contradictions.append(
                "Stock model does not support an upward signal."
            )

        if signals["financial_signal"] == 0:
            contradictions.append(
                "Financial model does not indicate strong financial health."
            )

        if signals["market_signal"] == 0:
            contradictions.append(
                "Market model does not indicate outperformance."
            )

        if signals["risk_signal"] == 1:
            contradictions.append(
                "Risk model indicates elevated risk."
            )

    elif recommendation == "SELL":

        if signals["stock_signal"] == 1:
            contradictions.append(
                "Stock model indicates an upward signal."
            )

        if signals["financial_signal"] == 1:
            contradictions.append(
                "Financial model indicates strong financial health."
            )

        if signals["market_signal"] == 1:
            contradictions.append(
                "Market model indicates outperformance."
            )

    elif recommendation == "HOLD":

        positive_count = (
            signals["stock_signal"]
            + signals["financial_signal"]
            + signals["market_signal"]
            + signals["sentiment_signal"]
        )

        if positive_count >= 3 and signals["risk_signal"] == 0:
            contradictions.append(
                "Multiple models support a stronger positive outlook."
            )

    if len(contradictions) == 0:
        verification_status = "PASS"
    else:
        verification_status = "REVIEW"

    return {
        "recommendation": recommendation,
        "verification_status": verification_status,
        "contradiction_count": len(contradictions),
        "contradictions": contradictions
    }

In [427]:
red_team_result = red_team_verification(
    synthesis_result
)

print(
    "Recommendation:",
    red_team_result["recommendation"]
)

print(
    "Verification:",
    red_team_result["verification_status"]
)

print(
    "Contradictions:",
    red_team_result["contradiction_count"]
)

for item in red_team_result["contradictions"]:
    print("-", item)

Recommendation: BUY
Verification: PASS
Contradictions: 0


In [428]:
with open(
    OUTPUT_PATH + "nvidia_red_team_result.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        red_team_result,
        f,
        indent=4
    )

print("Red-Team result saved")

Red-Team result saved


In [429]:
final_result = {
    "query": synthesis_result["query"] if "query" in synthesis_result else "Should I invest in NVIDIA?",
    "recommendation": synthesis_result["recommendation"],
    "confidence": synthesis_result["confidence"],
    "probabilities": synthesis_result["probabilities"],
    "signals": synthesis_result["signals"],
    "red_team": red_team_result
}

print("FINAL INVESTMENT DECISION")
print("-------------------------")
print("Recommendation:", final_result["recommendation"])
print("Confidence:", final_result["confidence"])
print("Verification:", final_result["red_team"]["verification_status"])

print("\nSignals:")
for key, value in final_result["signals"].items():
    print(key, ":", value)

FINAL INVESTMENT DECISION
-------------------------
Recommendation: BUY
Confidence: 0.71
Verification: PASS

Signals:
stock_signal : 1
financial_signal : 1
market_signal : 1
risk_signal : 0
sentiment_signal : 0


In [430]:
with open(
    OUTPUT_PATH + "nvidia_final_investment_decision.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_result,
        f,
        indent=4,
        default=str
    )

print("Final investment decision saved")

Final investment decision saved


In [431]:
def analyze_nvidia(query):

    print("=" * 60)
    print("NVIDIA FINANCIAL RESEARCH SYSTEM")
    print("=" * 60)

    print("\nQuery:")
    print(query)

    # Route query
    routed = route_to_model(query)

    print("\nCategory:", routed["category"])
    print("Model:", routed["model"])

    # RAG
    rag_results = retrieve_documents(
        query,
        rag_vectorizer,
        rag_matrix,
        rag_chunks,
        top_k=5
    )

    # Model synthesis
    synthesis = investment_synthesis_prediction(query)

    # Red-team
    verification = red_team_verification(synthesis)

    result = {
        "query": query,
        "category": routed["category"],
        "model": routed["model"],
        "synthesis": synthesis,
        "verification": verification,
        "evidence": rag_results[
            ["chunk_id", "Similarity", "text"]
        ].to_dict("records")
    }

    return result

In [432]:
final_analysis = analyze_nvidia(
    "Should I invest in NVIDIA?"
)

print("\n")
print("FINAL RECOMMENDATION:",
      final_analysis["synthesis"]["recommendation"])

print(
    "CONFIDENCE:",
    final_analysis["synthesis"]["confidence"]
)

print(
    "RED-TEAM:",
    final_analysis["verification"]["verification_status"]
)

NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
Should I invest in NVIDIA?

Category: research
Model: Research Model


FINAL RECOMMENDATION: BUY
CONFIDENCE: 0.71
RED-TEAM: PASS


In [433]:
test_questions = [
    "What are NVIDIA's biggest risks?",
    "How is NVIDIA financially?",
    "How does NVIDIA compare with AMD?",
    "What is NVIDIA's valuation?",
    "What is the outlook for NVIDIA stock?",
    "What is NVIDIA's competitive position?"
]

for query in test_questions:

    result = analyze_nvidia(query)

    print("\n" + "=" * 60)
    print("QUESTION:", query)
    print(
        "RECOMMENDATION:",
        result["synthesis"]["recommendation"]
    )

NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
What are NVIDIA's biggest risks?

Category: risk
Model: Risk Classification Model

QUESTION: What are NVIDIA's biggest risks?
RECOMMENDATION: BUY
NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
How is NVIDIA financially?

Category: financial
Model: Financial Analysis Model

QUESTION: How is NVIDIA financially?
RECOMMENDATION: BUY
NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
How does NVIDIA compare with AMD?

Category: market
Model: Market & Competitor Model

QUESTION: How does NVIDIA compare with AMD?
RECOMMENDATION: BUY
NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
What is NVIDIA's valuation?

Category: valuation
Model: Valuation Model

QUESTION: What is NVIDIA's valuation?
RECOMMENDATION: BUY
NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
What is the outlook for NVIDIA stock?

Category: stock
Model: Stock Prediction Model

QUESTION: What is the outlook for NVIDIA stock?
RECOMMENDATION: BUY
NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
What is NVIDIA's competitive positi

In [434]:
def generate_final_response(query):

    result = analyze_nvidia(query)

    synthesis = result["synthesis"]
    verification = result["verification"]

    response = []

    response.append(f"Question: {query}\n")

    response.append(
        f"Investment Recommendation: {synthesis['recommendation']}"
    )

    if synthesis["confidence"] is not None:
        response.append(
            f"Confidence: {synthesis['confidence']:.2f}"
        )

    response.append("\nModel Signals:")

    for key, value in synthesis["signals"].items():
        response.append(f"- {key}: {value}")

    response.append(
        f"\nRed-Team Verification: "
        f"{verification['verification_status']}"
    )

    if verification["contradictions"]:
        response.append("\nPoints requiring review:")

        for item in verification["contradictions"]:
            response.append(f"- {item}")

    response.append("\nEvidence:")

    evidence = result["evidence"]

    for row in evidence[:3]:

        text = str(row["text"]).replace("\n", " ")

        if len(text) > 300:
            text = text[:300] + "..."

        response.append(f"- {text}")

    return "\n".join(response)

In [435]:
print(
    generate_final_response(
        "Should I invest in NVIDIA?"
    )
)

NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
Should I invest in NVIDIA?

Category: research
Model: Research Model
Question: Should I invest in NVIDIA?

Investment Recommendation: BUY
Confidence: 0.71

Model Signals:
- stock_signal: 1
- financial_signal: 1
- market_signal: 1
- risk_signal: 0
- sentiment_signal: 0

Red-Team Verification: PASS

Evidence:
- to the extent that our perceived ability to consummate acquisitions is harmed, future acquisitions may be more difficult, complex or expensive. Our investments in companies could create volatility and fluctuations in our results. These investments may generate realized and unrealized gains or losses...
- and challenges in obtaining regulatory approvals. Our investments in private companies include early-stage companies still defining their strategic direction. Many of the securities in which we invest are non-marketable and illiquid at the time of our initial investment. To the extent any of the com...
- number of our customers and their da

In [436]:
questions = [
    "How is NVIDIA financially?",
    "What are NVIDIA's biggest risks?",
    "How does NVIDIA compare with AMD?",
    "What is NVIDIA's valuation?",
    "Should I invest in NVIDIA?"
]

for query in questions:

    print("\n" + "=" * 80)
    print(generate_final_response(query))


NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
How is NVIDIA financially?

Category: financial
Model: Financial Analysis Model
Question: How is NVIDIA financially?

Investment Recommendation: BUY
Confidence: 0.71

Model Signals:
- stock_signal: 1
- financial_signal: 1
- market_signal: 1
- risk_signal: 0
- sentiment_signal: 0

Red-Team Verification: PASS

Evidence:
- complying with our disclosure obligations under Regulation FD: NVIDIA Corporate Blog (blogs.nvidia.com/) NVIDIA Technical Blog (developer.nvidia.com/blog/) NVIDIA LinkedIn (linkedin.com/company/nvidia) NVIDIA Facebook (facebook.com/nvidia) NVIDIA Instagram (instagram.com/nvidia) NVIDIA X (x.com/nvid...
- (blogs.nvidia.com/) NVIDIA Technical Blog (developer.nvidia.com/blog/) NVIDIA LinkedIn (linkedin.com/company/nvidia) NVIDIA Facebook (facebook.com/nvidia) NVIDIA Instagram (instagram.com/nvidia) NVIDIA X (x.com/nvidia) NVIDIA Threads (threads.com/@nvidia) NVIDIA Investor Relations (investor.nvidia.c...
- Related Transactions, an

In [437]:
def classify_query(query):

    query_lower = query.lower()

    investment_keywords = [
        "should i invest",
        "should i buy",
        "should i sell",
        "invest in",
        "investment",
        "buy or sell",
        "worth buying",
        "good investment",
        "investing"
    ]

    for keyword in investment_keywords:

        if keyword in query_lower:
            return "investment", {"investment": 1}

    category_scores = {}

    for category, keywords in query_categories.items():

        score = 0

        for keyword in keywords:

            if keyword in query_lower:
                score += 1

        category_scores[category] = score

    priority = [
        "stock",
        "financial",
        "valuation",
        "risk",
        "market",
        "sentiment",
        "research"
    ]

    best_category = "research"
    best_score = 0

    for category in priority:

        score = category_scores.get(category, 0)

        if score > best_score:

            best_score = score
            best_category = category

    return best_category, category_scores

In [438]:
def route_to_model(query):

    category, scores = classify_query(query)

    model_map = {
        "investment": "Investment Synthesis Model",
        "stock": "Stock Prediction Model",
        "financial": "Financial Analysis Model",
        "market": "Market & Competitor Model",
        "risk": "Risk Classification Model",
        "sentiment": "News Sentiment Model",
        "research": "Research Model",
        "valuation": "Valuation Model"
    }

    return {
        "category": category,
        "model": model_map.get(
            category,
            "Research Model"
        ),
        "scores": scores
    }

In [439]:
def analyze_nvidia(query):

    print("=" * 60)
    print("NVIDIA FINANCIAL RESEARCH SYSTEM")
    print("=" * 60)

    print("\nQuery:")
    print(query)

    routed = route_to_model(query)

    print("\nCategory:", routed["category"])
    print("Model:", routed["model"])

    rag_results = retrieve_documents(
        query,
        rag_vectorizer,
        rag_matrix,
        rag_chunks,
        top_k=5
    )

    if routed["category"] == "investment":

        synthesis = investment_synthesis_prediction(query)

        verification = red_team_verification(
            synthesis
        )

    else:

        synthesis = investment_synthesis_prediction(query)

        verification = red_team_verification(
            synthesis
        )

    result = {
        "query": query,
        "category": routed["category"],
        "model": routed["model"],
        "synthesis": synthesis,
        "verification": verification,
        "evidence": rag_results[
            ["chunk_id", "Similarity", "text"]
        ].to_dict("records")
    }

    return result

In [440]:
final_analysis = analyze_nvidia(
    "Should I invest in NVIDIA?"
)

print("\nFINAL RECOMMENDATION:")
print(
    final_analysis["synthesis"]["recommendation"]
)

print("\nCONFIDENCE:")
print(
    final_analysis["synthesis"]["confidence"]
)

print("\nVERIFICATION:")
print(
    final_analysis["verification"]["verification_status"]
)

NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
Should I invest in NVIDIA?

Category: investment
Model: Investment Synthesis Model

FINAL RECOMMENDATION:
BUY

CONFIDENCE:
0.71

VERIFICATION:
PASS


In [441]:
print(
    generate_final_response(
        "Should I invest in NVIDIA?"
    )
)

NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
Should I invest in NVIDIA?

Category: investment
Model: Investment Synthesis Model
Question: Should I invest in NVIDIA?

Investment Recommendation: BUY
Confidence: 0.71

Model Signals:
- stock_signal: 1
- financial_signal: 1
- market_signal: 1
- risk_signal: 0
- sentiment_signal: 0

Red-Team Verification: PASS

Evidence:
- to the extent that our perceived ability to consummate acquisitions is harmed, future acquisitions may be more difficult, complex or expensive. Our investments in companies could create volatility and fluctuations in our results. These investments may generate realized and unrealized gains or losses...
- and challenges in obtaining regulatory approvals. Our investments in private companies include early-stage companies still defining their strategic direction. Many of the securities in which we invest are non-marketable and illiquid at the time of our initial investment. To the extent any of the com...
- number of our customer

In [442]:
def nvidia_chat():

    print("=" * 70)
    print("NVIDIA FINANCIAL RESEARCH ASSISTANT")
    print("Type 'exit' to stop")
    print("=" * 70)

    while True:

        query = input("\nYou: ")

        if query.lower().strip() == "exit":
            print("Session ended.")
            break

        try:

            answer = generate_final_response(query)

            print("\nAssistant:")
            print(answer)

        except Exception as e:

            print("\nError:")
            print(str(e))

In [443]:
nvidia_chat()

NVIDIA FINANCIAL RESEARCH ASSISTANT
Type 'exit' to stop



You:  Hi


NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
Hi

Category: research
Model: Research Model

Assistant:
Question: Hi

Investment Recommendation: BUY
Confidence: 0.71

Model Signals:
- stock_signal: 1
- financial_signal: 1
- market_signal: 1
- risk_signal: 0
- sentiment_signal: 0

Red-Team Verification: PASS

Evidence:
- August 26, 2026 NVIDIA Corporation By: /s/ Colette M. Kress Colette M. Kress Executive Vice President and Chief Financial Officer (Duly Authorized Officer and Principal Financial Officer) 43
- harmed if we are unsuccessful or if we are prohibited from making or selling our products. From time to time, we are involved in lawsuits or other legal proceedings alleging patent infringement or other IP rights violations by us, our employees or parties that we have agreed to indemnify. An unfavor...
- cash tax payments, results of operations, and financial condition. Our future effective tax rate may also be affected by a variety of factors, including changes in our business or statu


You:  Hi


NVIDIA FINANCIAL RESEARCH SYSTEM

Query:
Hi

Category: research
Model: Research Model

Assistant:
Question: Hi

Investment Recommendation: BUY
Confidence: 0.71

Model Signals:
- stock_signal: 1
- financial_signal: 1
- market_signal: 1
- risk_signal: 0
- sentiment_signal: 0

Red-Team Verification: PASS

Evidence:
- August 26, 2026 NVIDIA Corporation By: /s/ Colette M. Kress Colette M. Kress Executive Vice President and Chief Financial Officer (Duly Authorized Officer and Principal Financial Officer) 43
- harmed if we are unsuccessful or if we are prohibited from making or selling our products. From time to time, we are involved in lawsuits or other legal proceedings alleging patent infringement or other IP rights violations by us, our employees or parties that we have agreed to indemnify. An unfavor...
- cash tax payments, results of operations, and financial condition. Our future effective tax rate may also be affected by a variety of factors, including changes in our business or statu

KeyboardInterrupt: Interrupted by user

In [444]:
system_files = [
    "nvidia_investment_synthesis_model.pkl",
    "nvidia_investment_synthesis_encoder.pkl",
    "nvidia_red_team_result.json",
    "nvidia_final_investment_decision.json",
    "nvidia_investment_synthesis_result.json"
]

for file in system_files:

    path = OUTPUT_PATH + file

    print(
        file,
        "->",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

nvidia_investment_synthesis_model.pkl -> EXISTS
nvidia_investment_synthesis_encoder.pkl -> EXISTS
nvidia_red_team_result.json -> EXISTS
nvidia_final_investment_decision.json -> EXISTS
nvidia_investment_synthesis_result.json -> EXISTS


In [445]:
app_code = r'''
import streamlit as st
import json
import os

st.set_page_config(
    page_title="NVIDIA Financial Research",
    page_icon="N",
    layout="wide"
)

st.title("NVIDIA Financial Research System")
st.caption("AI-powered financial research and due-diligence engine")

query = st.text_input(
    "Ask a question about NVIDIA",
    placeholder="Should I invest in NVIDIA?"
)

if st.button("Analyze") and query.strip():

    try:

        result = generate_final_response(query)

        st.markdown("### Analysis")

        st.write(result)

    except Exception as e:

        st.error(str(e))
'''

app_path = OUTPUT_PATH + "nvidia_app.py"

with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Created:", app_path)

Created: ../outputs/nvidia_app.py


In [446]:
print(
    os.path.exists(
        OUTPUT_PATH + "nvidia_app.py"
    )
)

print(
    OUTPUT_PATH + "nvidia_app.py"
)

True
../outputs/nvidia_app.py


In [447]:
import inspect
import os

engine_path = OUTPUT_PATH + "nvidia_engine.py"

function_names = [
    "classify_query",
    "route_to_model",
    "retrieve_documents",
    "stock_model_prediction",
    "financial_model_prediction",
    "market_model_prediction",
    "risk_prediction",
    "sentiment_prediction",
    "valuation_model_prediction",
    "research_prediction",
    "generate_model_output",
    "collect_current_model_signals",
    "investment_synthesis_prediction",
    "red_team_verification",
    "analyze_nvidia",
    "generate_final_response"
]

with open(engine_path, "w", encoding="utf-8") as f:

    f.write("""
import os
import json
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = os.path.dirname(
    os.path.dirname(os.path.abspath(__file__))
)

OUTPUT_PATH = PROJECT_ROOT + "/outputs/"

""")

    # Load saved artifacts
    f.write("""
# =========================
# LOAD SAVED MODELS
# =========================

stock_model = joblib.load(
    OUTPUT_PATH + "nvidia_logistic_final_model.pkl"
)

stock_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_logistic_scaler.pkl"
)

sentiment_model = joblib.load(
    OUTPUT_PATH + "nvidia_financial_sentiment_model.pkl"
)

sentiment_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_sentiment_tfidf_vectorizer.pkl"
)

risk_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_risk_model.pkl"
)

risk_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_risk_tfidf_vectorizer.pkl"
)

market_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_market_competitor_model.pkl"
)

market_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_market_competitor_scaler.pkl"
)

research_model = joblib.load(
    OUTPUT_PATH + "nvidia_research_topic_model.pkl"
)

research_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_research_tfidf_vectorizer.pkl"
)

best_financial_model = joblib.load(
    OUTPUT_PATH + "best_nvidia_financial_model.pkl"
)

financial_scaler = joblib.load(
    OUTPUT_PATH + "nvidia_financial_scaler.pkl"
)

investment_synthesis_model = joblib.load(
    OUTPUT_PATH + "nvidia_investment_synthesis_model.pkl"
)

synthesis_encoder = joblib.load(
    OUTPUT_PATH + "nvidia_investment_synthesis_encoder.pkl"
)

rag_vectorizer = joblib.load(
    OUTPUT_PATH + "nvidia_rag_tfidf_vectorizer.pkl"
)

rag_matrix = load_npz(
    OUTPUT_PATH + "nvidia_rag_matrix.npz"
)

rag_chunks = pd.read_csv(
    OUTPUT_PATH + "nvidia_rag_chunks.csv"
)

""")

    # Write function source
    for name in function_names:

        if name in globals():

            try:
                source = inspect.getsource(globals()[name])

                f.write("\n\n")
                f.write(source)
                f.write("\n")

            except Exception as e:

                print(
                    "Could not export:",
                    name,
                    "|",
                    e
                )

print("Engine created:")
print(engine_path)

Engine created:
../outputs/nvidia_engine.py


In [448]:
import os

engine_path = OUTPUT_PATH + "nvidia_engine.py"

print(
    "Exists:",
    os.path.exists(engine_path)
)

print(
    "Size:",
    os.path.getsize(engine_path),
    "bytes"
)

Exists: True
Size: 15553 bytes


In [449]:
%pip install -q sentence-transformers faiss-cpu

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
import faiss

print("Deep Learning RAG libraries loaded")

/Users/naitikdonda/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/naitikdonda/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
rag_chunks_dl = pd.read_csv(
    DATA_PATH + "nvidia_rag_chunks.csv"
)

print("Chunks:", len(rag_chunks_dl))
print("Columns:", rag_chunks_dl.columns.tolist())

rag_chunks_dl.head()